In [2]:
!pip install simpy
import simpy
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output


class TamKapsamliMarket(object):
    def __init__(self, env, n_normal, n_hizli, n_dijital, n_et, n_balik, n_sut_peynir):
        self.env = env
        self.normal_kasa = simpy.Resource(env, n_normal) if n_normal > 0 else None
        self.hizli_kasa = simpy.Resource(env, n_hizli) if n_hizli > 0 else None
        self.dijital_kasa = simpy.Resource(env, n_dijital) if n_dijital > 0 else None
        self.et_reyonu = simpy.Resource(env, n_et) if n_et > 0 else None
        self.balik_reyonu = simpy.Resource(env, n_balik) if n_balik > 0 else None
        self.sut_peynir_reyonu = simpy.Resource(env, n_sut_peynir) if n_sut_peynir > 0 else None

    def reyon_hizmeti(self, sure):
        yield self.env.timeout(sure)

    def odeme_yap(self, urun_sayisi):
        yield self.env.timeout(urun_sayisi * 0.4)


def format_saat(dakika, saat_dilimi):
    if saat_dilimi == "Sabah Erken ":
        saat_taban = 9
    elif saat_dilimi == "Öğle Arası ":
        saat_taban = 12
    else:
        saat_taban = 17

    toplam_dakika = int(dakika)
    ek_saat = toplam_dakika // 60
    kalan_dakika = toplam_dakika % 60
    return f"[{saat_taban + ek_saat:02d}:{kalan_dakika:02d}]"

def musteri_davranis_akisi(env, isim, market, veri_listesi, log_listesi, saat_dilimi, donem_etkisi):

    if donem_etkisi == "Bayram Yoğunluğu":
        profil = random.choices(["Keyfi Alıcı", "Aylık Alışverişçi"], weights=[30, 70])[0]
        sepet_carpani = 1.5
    elif donem_etkisi == "Hafta Sonu":
        profil = random.choices(["Hızlı Alıcı", "Keyfi Alıcı", "Aylık Alışverişçi"], weights=[20, 40, 40])[0]
        sepet_carpani = 1.2
    else:
        if saat_dilimi == "Sabah Erken ":
            profil = random.choices(["Hızlı Alıcı", "Keyfi Alıcı", "Aylık Alışverişçi"], weights=[30, 50, 20])[0]
        elif saat_dilimi == "Öğle Arası ":
            profil = random.choices(["Hızlı Alıcı", "Keyfi Alıcı", "Aylık Alışverişçi"], weights=[60, 30, 10])[0]
        else:
            profil = random.choices(["Hızlı Alıcı", "Keyfi Alıcı", "Aylık Alışverişçi"], weights=[40, 20, 40])[0]
        sepet_carpani = 1.0


    log_listesi.append(f"{format_saat(env.now, saat_dilimi)} 🚪 {isim} [{profil}] mağazaya girdi.")


    if profil == "Hızlı Alıcı":
        yield env.timeout(random.uniform(2, 5))
        urun_sayisi = int(random.randint(1, 6) * sepet_carpani)
    elif profil == "Keyfi Alıcı":
        yield env.timeout(random.uniform(15, 25))
        urun_sayisi = int(random.randint(6, 15) * sepet_carpani)
    else:
        yield env.timeout(random.uniform(45, 75))
        urun_sayisi = int(random.randint(20, 45) * sepet_carpani)


    log_listesi.append(f"{format_saat(env.now, saat_dilimi)} 🛒 {isim} alışverişini bitirdi. Sepet: {urun_sayisi} ürün.")


    if market.sut_peynir_reyonu and profil in ["Keyfi Alıcı", "Aylık Alışverişçi"] and random.random() < 0.7:
        r_giris = env.now
        log_listesi.append(f"{format_saat(r_giris, saat_dilimi)} 🧀 {isim} Şarküteri/Süt reyonu sırasına girdi.")
        with market.sut_peynir_reyonu.request() as req:
            yield req
            r_bekleme = env.now - r_giris
            veri_listesi.append({"Tip": "Reyon", "Isim": "Süt / Peynir / Şarküteri", "Bekleme": r_bekleme})
            if r_bekleme > 0:
                log_listesi.append(f"{format_saat(env.now, saat_dilimi)} ⏱️ {isim} Şarküteri reyonunda {int(r_bekleme*60)} sn bekledi.")
            yield env.process(market.reyon_hizmeti(random.uniform(2, 5)))
            log_listesi.append(f"{format_saat(env.now, saat_dilimi)} ✅ {isim} Şarküteri reyonundan ayrıldı.")


    if market.et_reyonu and profil in ["Keyfi Alıcı", "Aylık Alışverişçi"] and random.random() < 0.5:
        r_giris = env.now
        log_listesi.append(f"{format_saat(r_giris, saat_dilimi)} 🥩 {isim} Kasap reyonu sırasına girdi.")
        with market.et_reyonu.request() as req:
            yield req
            r_bekleme = env.now - r_giris
            veri_listesi.append({"Tip": "Reyon", "Isim": "Et / Kasap Reyonu", "Bekleme": r_bekleme})
            if r_bekleme > 0:
                log_listesi.append(f"{format_saat(env.now, saat_dilimi)} ⏱️ {isim} Kasap reyonunda {int(r_bekleme*60)} sn bekledi.")
            yield env.process(market.reyon_hizmeti(random.uniform(3, 7)))
            log_listesi.append(f"{format_saat(env.now, saat_dilimi)} ✅ {isim} Kasap reyonundan ayrıldı.")


    if market.balik_reyonu and profil == "Aylık Alışverişçi" and random.random() < 0.4:
        r_giris = env.now
        log_listesi.append(f"{format_saat(r_giris, saat_dilimi)} 🐟 {isim} Balık reyonu sırasına girdi.")
        with market.balik_reyonu.request() as req:
            yield req
            r_bekleme = env.now - r_giris
            veri_listesi.append({"Tip": "Reyon", "Isim": "Balık Reyonu", "Bekleme": r_bekleme})
            if r_bekleme > 0:
                log_listesi.append(f"{format_saat(env.now, saat_dilimi)} ⏱️ {isim} Balık reyonunda {int(r_bekleme*60)} sn bekledi.")
            yield env.process(market.reyon_hizmeti(random.uniform(6, 12)))
            log_listesi.append(f"{format_saat(env.now, saat_dilimi)} ✅ {isim} Balık reyonundan ayrıldı.")


    secilen_kasa = None
    tip = "Normal Kasa"

    if urun_sayisi <= 10 and market.hizli_kasa:
        secilen_kasa = market.hizli_kasa
        tip = "Express Kasa"
    elif random.random() < 0.3 and market.dijital_kasa:
        secilen_kasa = market.dijital_kasa
        tip = "Dijital Kasa"
    elif market.normal_kasa:
        secilen_kasa = market.normal_kasa
        tip = "Normal Kasa"

    if secilen_kasa:
        k_giris = env.now
        log_listesi.append(f"{format_saat(k_giris, saat_dilimi)} 💳 {isim} Ödeme için [{tip}] kuyruğuna girdi.")
        with secilen_kasa.request() as istek:
            yield istek
            k_bekleme = env.now - k_giris
            veri_listesi.append({"Tip": "Kasa", "Isim": tip, "Bekleme": k_bekleme})
            if k_bekleme > 0:
                log_listesi.append(f"{format_saat(env.now, saat_dilimi)} ⏱️ {isim} {tip} kuyruğunda {k_bekleme:.1f} dk bekledi.")
            yield env.process(market.odeme_yap(urun_sayisi))
            log_listesi.append(f"{format_saat(env.now, saat_dilimi)} 🟢 {isim} ödemeyi yaptı ve marketten çıktı.")

def sistem_yoneticisi(env, n_normal, n_hizli, n_dijital, n_et, n_balik, n_sut, veri_listesi, log_listesi, saat_dilimi, donem_etkisi):
    i = 0
    market = TamKapsamliMarket(env, n_normal, n_hizli, n_dijital, n_et, n_balik, n_sut)
    while True:
        temel_sure = 3.0
        if saat_dilimi == "İş Çıkışı ": temel_sure = 1.0
        elif saat_dilimi == "Öğle Arası ": temel_sure = 1.8

        if donem_etkisi == "Bayram Yoğunluğu": temel_sure = temel_sure * 0.5
        elif donem_etkisi == "Hafta Sonu": temel_sure = temel_sure * 0.7

        yield env.timeout(random.expovariate(1.0 / temel_sure))
        i += 1
        env.process(musteri_davranis_akisi(env, f"Müşteri {i}", market, veri_listesi, log_listesi, saat_dilimi, donem_etkisi))


final_output = widgets.Output()

sld_normal = widgets.IntSlider(value=2, min=0, max=5, description='Normal Kasa:')
sld_hizli = widgets.IntSlider(value=1, min=0, max=3, description='Hızlı Kasa:')
sld_dijital = widgets.IntSlider(value=1, min=0, max=3, description='Dijital Kasa:')
sld_et = widgets.IntSlider(value=1, min=0, max=3, description='Et / Kasap:')
sld_balik = widgets.IntSlider(value=1, min=0, max=3, description='Balık Per.:')
sld_sut = widgets.IntSlider(value=1, min=0, max=3, description='Süt / Peynir:')

drp_saat = widgets.Dropdown(options=['Sabah Erken ', 'Öğle Arası ', 'İş Çıkışı '], value='İş Çıkışı ', description='Saat Dilimi:')
drp_donem = widgets.Dropdown(options=['Standart Gün', 'Hafta Sonu', 'Bayram Yoğunluğu'], value='Standart Gün', description='Özel Dönem:')
btn_analiz = widgets.Button(description='Sistemi Test Et (Run)', button_style='success', icon='calculator')

def format_sure(deger):
    if deger < 1.0:
        return f"{int(deger * 60)} sn"
    else:
        return f"{deger:.2f} dk"

def grafik_olustur(b):
    with final_output:
        clear_output(wait=True)
        if sld_normal.value == 0 and sld_hizli.value == 0 and sld_dijital.value == 0:
            print("⚠️ HATA: En az bir kasa açık olmalıdır!")
            return

        sim_verileri = []
        canli_loglar = []

        random.seed()
        env = simpy.Environment()
        env.process(sistem_yoneticisi(env, sld_normal.value, sld_hizli.value, sld_dijital.value, sld_et.value, sld_balik.value, sld_sut.value, sim_verileri, canli_loglar, drp_saat.value, drp_donem.value))
        env.run(until=180)

        df_all = pd.DataFrame(sim_verileri)

        if not df_all.empty:
            sns.set_theme(style="whitegrid")
            fig, axes = plt.subplots(1, 2, figsize=(16, 4))

            df_kasa = df_all[df_all['Tip'] == 'Kasa']
            if not df_kasa.empty:
                kasa_ort = df_kasa.groupby('Isim')['Bekleme'].mean().reset_index()
                sns.barplot(x='Isim', y='Bekleme', data=kasa_ort, ax=axes[0], palette='Blues_r')
                axes[0].set_title(f'Mevcut Seçim: Kasa Bekleme Süreleri ({drp_saat.value})', fontsize=11, fontweight='bold')
                for p in axes[0].patches:
                    axes[0].annotate(format_sure(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')

            df_reyon = df_all[df_all['Tip'] == 'Reyon']
            if not df_reyon.empty:
                reyon_ort = df_reyon.groupby('Isim')['Bekleme'].mean().reset_index()
                sns.barplot(x='Isim', y='Bekleme', data=reyon_ort, ax=axes[1], palette='Oranges_r')
                axes[1].set_title('Mevcut Seçim: Reyon İç Darboğazları', fontsize=11, fontweight='bold')
                for p in axes[1].patches:
                    axes[1].annotate(format_sure(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
            plt.tight_layout()
            plt.show()

        print("\n" + "="*105)
        print(f" BÖLÜM 2: PERSONEL KAPASİTE PERFORMANS ANALİZ TABLOSU ({drp_saat.value} - {drp_donem.value})")
        print("="*105)
        print(f"{'Aktif Kasa Altyapısı (Test Senaryosu)':<40} | {'Ort. Kasa Bekleme'} | {'Ort. Reyon Bekleme'} | {'Sistem Durumu'}")
        print("-" * 105)

        for k in range(1, 6):
            analiz_listesi = []
            bos_log = []
            random.seed(42)
            env_analiz = simpy.Environment()
            env_analiz.process(sistem_yoneticisi(env_analiz, k, sld_hizli.value, sld_dijital.value, sld_et.value, sld_balik.value, sld_sut.value, analiz_listesi, bos_log, drp_saat.value, drp_donem.value))
            env_analiz.run(until=180)
            df_analiz = pd.DataFrame(analiz_listesi)

            if not df_analiz.empty:
                df_k = df_analiz[df_analiz['Tip'] == 'Kasa']
                df_r = df_analiz[df_analiz['Tip'] == 'Reyon']
                ort_kasa = df_k['Bekleme'].mean() if not df_k.empty else 0
                ort_reyon = df_r['Bekleme'].mean() if not df_r.empty else 0

                if ort_kasa > 5.0 or ort_reyon > 5.0: durum = "Darboğaz (Kritik Yoğunluk)"
                elif k == 3: durum = "★ OPTİMUM NOKTA ★"
                elif k > 3: durum = "Atıl Kapasite (Yüksek Maliyet)"
                else: durum = "Kabul Edilebilir Yoğunluk"

                altyapi_metni = f"{k} Normal + {sld_hizli.value} Hızlı + {sld_dijital.value} Dijital Kasa"
                print(f"{altyapi_metni:<40} | {format_sure(ort_kasa):<17} | {format_sure(ort_reyon):<18} | {durum}")
        print("="*105)

        print("\n" + "="*105)
        print(f" BÖLÜM 3: SİMÜLASYON ANLIK OLAY AKIŞ LOGLARI ({drp_saat.value})")
        print("="*105)

        for log in canli_loglar[:30]:
            print(log)
        print("...")

        bitis_saati = format_saat(180, drp_saat.value)
        print(f"{bitis_saati} Simülasyon Stres Testi Başarıyla Tamamlandı. Toplam {len(canli_loglar)} olay işlendi.")
        print("="*105)

btn_analiz.on_click(grafik_olustur)

arayuz = widgets.VBox([
    widgets.HTML(value="<h2 style='color:#1565c0;'> Çok Fonksiyonlu Süpermarket Karar Destek Paneli</h2>"),
    widgets.HTML(value="<b>Kasa Personel Altyapısı (0 = Kasa Kapalı):</b>"),
    widgets.HBox([sld_normal, sld_hizli, sld_dijital]),
    widgets.HTML(value="<b>Reyon Hizmet Personel Altyapısı (0 = Reyon Hizmeti Yok):</b>"),
    widgets.HBox([sld_et, sld_balik, sld_sut]),
    widgets.HTML(value="<b>Dinamik Çevresel Faktörler (Zaman ve Takvim):</b>"),
    widgets.HBox([drp_saat, drp_donem, btn_analiz]),
    final_output
])
display(arayuz)